In [1]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# 0. Create directory
os.makedirs('../models/sms', exist_ok=True)

# 1. Load Dataset (handles latin-1 encoding common in SMS dataset)
try:
    df_sms = pd.read_csv('../dataset/sms/sms_spam.csv', encoding='latin-1')
except Exception:
    df_sms = pd.read_csv('../dataset/sms/sms_spam.csv')

# Standardize columns
if 'v1' in df_sms.columns and 'v2' in df_sms.columns:
    df_sms = df_sms[['v1', 'v2']].rename(columns={'v1': 'label', 'v2': 'text'})
elif 'Category' in df_sms.columns and 'Message' in df_sms.columns:
    df_sms = df_sms[['Category', 'Message']].rename(columns={'Category': 'label', 'Message': 'text'})

df_sms['text'] = df_sms['text'].fillna('')
df_sms['target'] = df_sms['label'].apply(lambda x: 1 if str(x).strip().lower() in ['spam', 'phishing', '1', 1] else 0)

print("=" * 50)
print(f"SMS Dataset Shape: {df_sms.shape[0]} rows")
print(f"Legitimate (0): {(df_sms['target'] == 0).sum()} | Spam/Phishing (1): {(df_sms['target'] == 1).sum()}")
print("=" * 50)

# 2. Split
X_train, X_test, y_train, y_test = train_test_split(
    df_sms['text'], df_sms['target'], test_size=0.2, random_state=42, stratify=df_sms['target']
)

# 3. TF-IDF
tfidf_sms = TfidfVectorizer(max_features=3000, stop_words='english', sublinear_tf=True)
X_train_tfidf = tfidf_sms.fit_transform(X_train)
X_test_tfidf = tfidf_sms.transform(X_test)

joblib.dump(tfidf_sms, '../models/sms/tfidf_vectorizer.pkl')
print("✅ Saved SMS TF-IDF Vectorizer")

# 4. Train Classifiers
classifiers = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=500, random_state=42),
    "Multinomial Naive Bayes": MultinomialNB(),
    "Random Forest": RandomForestClassifier(n_estimators=50, max_depth=20, class_weight='balanced', random_state=42, n_jobs=-1)
}

results_sms = []

for name, clf in classifiers.items():
    clf.fit(X_train_tfidf, y_train)
    y_pred = clf.predict(X_test_tfidf)
    y_proba = clf.predict_proba(X_test_tfidf)[:, 1] if hasattr(clf, "predict_proba") else y_pred
    
    results_sms.append({
        "Model": name,
        "Precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "F1-Score": round(f1_score(y_test, y_pred), 4),
        "ROC-AUC": round(roc_auc_score(y_test, y_proba), 4)
    })
    
    joblib.dump(clf, f"../models/sms/{name.lower().replace(' ', '_')}.pkl")

print("\n" + "="*50)
print("     MODULE 3 (PART A): SMS MODEL COMPARISON")
print("="*50)
print(pd.DataFrame(results_sms).to_string(index=False))

SMS Dataset Shape: 5572 rows
Legitimate (0): 4825 | Spam/Phishing (1): 747
✅ Saved SMS TF-IDF Vectorizer

     MODULE 3 (PART A): SMS MODEL COMPARISON
                  Model  Precision  Recall  F1-Score  ROC-AUC
    Logistic Regression     0.9444  0.9128    0.9283   0.9868
Multinomial Naive Bayes     0.9840  0.8255    0.8978   0.9896
          Random Forest     0.9612  0.8322    0.8921   0.9758


In [2]:
# 0. Create directory
os.makedirs('../models/url', exist_ok=True)

# 1. Load URL Dataset
df_url = pd.read_csv('../dataset/url/malicious_urls.csv')

url_col = [c for c in df_url.columns if c.lower() in ['url', 'link', 'domain']][0]
label_col = [c for c in df_url.columns if c.lower() in ['label', 'type', 'class', 'target']][0]

df_url[url_col] = df_url[url_col].fillna('')
df_url['target'] = df_url[label_col].apply(lambda x: 0 if str(x).strip().lower() in ['benign', 'good', '0', 0] else 1)

print("\n" + "=" * 50)
print(f"URL Dataset Shape: {df_url.shape[0]} rows")
print(f"Safe URLs (0): {(df_url['target'] == 0).sum()} | Malicious URLs (1): {(df_url['target'] == 1).sum()}")
print("=" * 50)

# Sample if dataset is exceptionally large (>100k) to optimize training speed
if len(df_url) > 100000:
    df_url = df_url.sample(n=100000, random_state=42).reset_index(drop=True)
    print(f"Subsampled to 100,000 records for fast training.")

# 2. Split
X_train_u, X_test_u, y_train_u, y_test_u = train_test_split(
    df_url[url_col], df_url['target'], test_size=0.2, random_state=42, stratify=df_url['target']
)

# 3. Character N-Gram TF-IDF Vectorizer (Optimal for URLs)
tfidf_url = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_features=3000)
X_train_u_tfidf = tfidf_url.fit_transform(X_train_u)
X_test_u_tfidf = tfidf_url.transform(X_test_u)

joblib.dump(tfidf_url, '../models/url/tfidf_vectorizer.pkl')
print("✅ Saved URL TF-IDF Vectorizer")

# 4. Train Classifiers
classifiers_url = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=500, random_state=42),
    "Multinomial Naive Bayes": MultinomialNB(),
    "Random Forest": RandomForestClassifier(n_estimators=50, max_depth=20, class_weight='balanced', random_state=42, n_jobs=-1)
}

results_url = []

for name, clf in classifiers_url.items():
    clf.fit(X_train_u_tfidf, y_train_u)
    y_pred = clf.predict(X_test_u_tfidf)
    y_proba = clf.predict_proba(X_test_u_tfidf)[:, 1] if hasattr(clf, "predict_proba") else y_pred
    
    results_url.append({
        "Model": name,
        "Precision": round(precision_score(y_test_u, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_test_u, y_pred), 4),
        "F1-Score": round(f1_score(y_test_u, y_pred), 4),
        "ROC-AUC": round(roc_auc_score(y_test_u, y_proba), 4)
    })
    
    joblib.dump(clf, f"../models/url/{name.lower().replace(' ', '_')}.pkl")

print("\n" + "="*50)
print("     MODULE 3 (PART B): MALICIOUS URL MODEL COMPARISON")
print("="*50)
print(pd.DataFrame(results_url).to_string(index=False))


URL Dataset Shape: 651191 rows
Safe URLs (0): 428103 | Malicious URLs (1): 223088
Subsampled to 100,000 records for fast training.
✅ Saved URL TF-IDF Vectorizer

     MODULE 3 (PART B): MALICIOUS URL MODEL COMPARISON
                  Model  Precision  Recall  F1-Score  ROC-AUC
    Logistic Regression     0.9271  0.9287    0.9279   0.9863
Multinomial Naive Bayes     0.8929  0.7694    0.8265   0.9550
          Random Forest     0.9655  0.9000    0.9316   0.9856


In [3]:
# Load trained SMS model
lr_sms = joblib.load('../models/sms/logistic_regression.pkl')
vec_sms = joblib.load('../models/sms/tfidf_vectorizer.pkl')

# Load trained URL model
lr_url = joblib.load('../models/url/logistic_regression.pkl')
vec_url = joblib.load('../models/url/tfidf_vectorizer.pkl')

def test_sms(text):
    vec = vec_sms.transform([text])
    pred = lr_sms.predict(vec)[0]
    prob = lr_sms.predict_proba(vec)[0][1]
    print(f"SMS: '{text}' --> {'🚨 SPAM/PHISHING' if pred == 1 else '✅ SAFE'} (Risk: {prob*100:.2f}%)")

def test_url(url_str):
    vec = vec_url.transform([url_str])
    pred = lr_url.predict(vec)[0]
    prob = lr_url.predict_proba(vec)[0][1]
    print(f"URL: '{url_str}' --> {'🚨 MALICIOUS/PHISHING' if pred == 1 else '✅ SAFE'} (Risk: {prob*100:.2f}%)")

print("\n--- Testing Module 3 Predictions ---")
test_sms("URGENT! You have won $1000 cash prize. Claim now at http://free-cash.xyz or call 800-12345")
test_sms("Hey, are we still meeting for lunch today at 1 PM?")
test_url("http://login-verify-account-security-update.com/signin")
test_url("https://www.google.com")


--- Testing Module 3 Predictions ---
SMS: 'URGENT! You have won $1000 cash prize. Claim now at http://free-cash.xyz or call 800-12345' --> 🚨 SPAM/PHISHING (Risk: 99.66%)
SMS: 'Hey, are we still meeting for lunch today at 1 PM?' --> ✅ SAFE (Risk: 7.15%)
URL: 'http://login-verify-account-security-update.com/signin' --> 🚨 MALICIOUS/PHISHING (Risk: 90.70%)
URL: 'https://www.google.com' --> 🚨 MALICIOUS/PHISHING (Risk: 100.00%)


In [4]:
from urllib.parse import urlparse

# Trusted domain whitelist for popular benign root domains
TRUSTED_DOMAINS = {
    "google.com", "www.google.com", "youtube.com", "www.youtube.com",
    "github.com", "microsoft.com", "linkedin.com", "amazon.com",
    "wikipedia.org", "apple.com", "yahoo.com"
}

def test_url_hybrid(url_str):
    # Parse root domain
    parsed = urlparse(url_str if url_str.startswith(('http://', 'https://')) else 'http://' + url_str)
    domain = parsed.netloc.lower()
    
    # 1. Whitelist Check
    if domain in TRUSTED_DOMAINS or domain.replace("www.", "") in TRUSTED_DOMAINS:
        print(f"URL: '{url_str}' --> ✅ SAFE (Trusted Domain Whitelist) (Risk: 0.00%)")
        return
        
    # 2. ML Model Inference
    vec = vec_url.transform([url_str])
    pred = lr_url.predict(vec)[0]
    prob = lr_url.predict_proba(vec)[0][1]
    
    status = "🚨 MALICIOUS/PHISHING" if pred == 1 else "✅ SAFE"
    print(f"URL: '{url_str}' --> {status} (Risk: {prob*100:.2f}%)")

print("\n--- Testing Fixed Hybrid URL Engine ---")
test_url_hybrid("http://login-verify-account-security-update.com/signin")
test_url_hybrid("https://www.google.com")
test_url_hybrid("https://github.com")


--- Testing Fixed Hybrid URL Engine ---
URL: 'http://login-verify-account-security-update.com/signin' --> 🚨 MALICIOUS/PHISHING (Risk: 90.70%)
URL: 'https://www.google.com' --> ✅ SAFE (Trusted Domain Whitelist) (Risk: 0.00%)
URL: 'https://github.com' --> ✅ SAFE (Trusted Domain Whitelist) (Risk: 0.00%)
